In [ ]:
# 🖼️ CLIP Image Filtering Notebook

Bu notebook, CLIP embeddings kullanarak resimleri benzerliklerine göre filtreler.

**Amaç:** Referans resimlere **benzemeyen** (unique) resimleri bulmak.

**İş Akışı:**
1. Google Drive'ı mount et
2. `first-50/` klasöründeki referans resimlerin embeddinglerini al
3. `raw-images/` altındaki klasörleri tara ve her klasörden belirli sayıda resim örnekle
4. Her resmi referans embeddinglerle karşılaştır
5. Threshold **altında** benzerlik gösteren (yani benzemeyen) resimleri `filtered-images/` klasörüne kaydet

## 1. Kurulum & Bağımlılıklar

In [ ]:
# CLIP ve gerekli kütüphaneleri yükle
!pip install -q git+https://github.com/openai/CLIP.git ftfy regex tqdm pandas pyarrow

In [ ]:
import torch
import clip
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
import shutil
import random
import os

# GPU kontrolü
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Google Drive Mount & Yollar

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Proje yolları
BASE_PATH = Path("/content/drive/MyDrive/furniture-retrieval")
FIRST50_PATH = BASE_PATH / "first-50"
RAW_IMAGES_PATH = BASE_PATH / "raw-images"
FILTERED_PATH = BASE_PATH / "filtered-images"

# Local cache (Drive IO'yu hızlandırmak için)
CACHE_PATH = Path("/content/cache")

# Klasörleri oluştur
FILTERED_PATH.mkdir(parents=True, exist_ok=True)
CACHE_PATH.mkdir(parents=True, exist_ok=True)

print(f"📁 Base Path: {BASE_PATH}")
print(f"📁 First-50 Path: {FIRST50_PATH}")
print(f"📁 Raw Images Path: {RAW_IMAGES_PATH}")
print(f"📁 Filtered Path: {FILTERED_PATH}")
print(f"📁 Cache Path: {CACHE_PATH}")

## 3. Konfigürasyon

In [ ]:
# ========== KONFİGÜRASYON ==========

# CLIP Model (ViT-L/14 daha iyi kalite, ViT-B/32 daha hızlı)
MODEL_NAME = "ViT-L/14"

# Batch size (A100 40GB için 256-512 kullanılabilir)
# A100: 256-512, V100: 128, T4: 32-64
BATCH_SIZE = 256

# Benzerlik eşiği - BU DEĞERDEN DÜŞÜK OLANLAR ALINACAK (benzemeyen resimler)
# < 0.60: Çok farklı resimler
# 0.60-0.70: Farklı resimler
# 0.70-0.80: Orta benzerlik
SIMILARITY_THRESHOLD = 0.70  # Bu değerden DÜŞÜK olanlar alınacak

# Desteklenen uzantılar
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

# Her klasörden alınacak resim sayısı
# Belirtilmeyen klasörler için DEFAULT_SAMPLE_COUNT kullanılır
# None = tüm resimleri al
TARGET_COUNTS = {
    # Örnek: "bedroom": 2000,
    # Örnek: "kitchen": 1500,
}
DEFAULT_SAMPLE_COUNT = 500  # Belirtilmeyen klasörler için

# Random seed (reproducibility için)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# A100 optimizasyonları
USE_TF32 = True  # A100'de TF32 ile daha hızlı matmul
NUM_WORKERS = 4  # DataLoader için worker sayısı

print("⚙️ Konfigürasyon:")
print(f"   Model: {MODEL_NAME}")
print(f"   Batch Size: {BATCH_SIZE} (A100 optimized)")
print(f"   Similarity Threshold: < {SIMILARITY_THRESHOLD} (benzemeyen resimler alınacak)")
print(f"   Default Sample Count: {DEFAULT_SAMPLE_COUNT}")
print(f"   TF32 Enabled: {USE_TF32}")

## 4. CLIP Model Yükleme

In [ ]:
# CLIP modelini yükle
print(f"🔄 CLIP modeli yükleniyor: {MODEL_NAME}...")

# A100 TF32 optimizasyonu (daha hızlı matmul, minimal accuracy loss)
if USE_TF32 and torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("⚡ TF32 aktif (A100 optimizasyonu)")

model, preprocess = clip.load(MODEL_NAME, device=device)
model.eval()

# FP16'ya çevir (A100'de daha hızlı)
model = model.half()
print(f"✅ Model yüklendi! (FP16 mode)")

# Embedding boyutunu al
with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224, dtype=torch.float16).to(device)
    embed_dim = model.encode_image(dummy).shape[1]
print(f"📐 Embedding boyutu: {embed_dim}")

## 5. Yardımcı Fonksiyonlar

In [ ]:
def get_image_files(folder_path: Path) -> list:
    """Klasördeki geçerli resim dosyalarını listele"""
    files = []
    for ext in VALID_EXTENSIONS:
        files.extend(folder_path.glob(f"*{ext}"))
        files.extend(folder_path.glob(f"*{ext.upper()}"))
    return sorted(files)


def load_image_safe(image_path: Path) -> Image.Image | None:
    """Resmi güvenli şekilde yükle, hata varsa None döndür"""
    try:
        img = Image.open(image_path).convert("RGB")
        return img
    except Exception as e:
        print(f"⚠️ Resim yüklenemedi: {image_path.name} - {e}")
        return None


def encode_images_batch(image_paths: list, batch_size: int = BATCH_SIZE) -> tuple:
    """
    Resimleri batch halinde encode et (A100 optimized).
    Returns: (embeddings tensor [N, embed_dim], valid_paths list)
    """
    all_embeddings = []
    valid_paths = []

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Encoding", leave=False):
        batch_paths = image_paths[i:i + batch_size]
        batch_tensors = []
        batch_valid_paths = []

        for path in batch_paths:
            img = load_image_safe(path)
            if img is not None:
                try:
                    tensor = preprocess(img)
                    batch_tensors.append(tensor)
                    batch_valid_paths.append(path)
                except Exception as e:
                    print(f"⚠️ Preprocess hatası: {path.name} - {e}")

        if batch_tensors:
            # FP16 için half() kullan
            batch_input = torch.stack(batch_tensors).to(device).half()
            with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
                embeddings = model.encode_image(batch_input)
                embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
            all_embeddings.append(embeddings.float().cpu())  # CPU'da float32'ye çevir
            valid_paths.extend(batch_valid_paths)

    if all_embeddings:
        return torch.cat(all_embeddings, dim=0), valid_paths
    return torch.empty(0, embed_dim), []


def compute_similarity(candidate_embeddings: torch.Tensor,
                       reference_embeddings: torch.Tensor) -> tuple:
    """
    Candidate ve reference embeddingler arasındaki max similarity hesapla.
    Returns: (max_scores [N], best_ref_indices [N])
    """
    # Cosine similarity = dot product (normalized vectors için)
    # candidate: [N, D], reference: [M, D] -> similarity: [N, M]
    similarity = candidate_embeddings @ reference_embeddings.T
    max_scores, best_indices = similarity.max(dim=1)
    return max_scores, best_indices


print("✅ Yardımcı fonksiyonlar tanımlandı (A100 optimized)")

## 6. Referans Resimlerin Embeddinglerini Al (first-50)

In [ ]:
# first-50 klasöründeki resimleri listele
first50_files = get_image_files(FIRST50_PATH)
print(f"📸 first-50 klasöründe {len(first50_files)} resim bulundu")

if len(first50_files) == 0:
    raise ValueError("❌ first-50 klasöründe resim bulunamadı! Lütfen resimleri ekleyin.")

In [ ]:
# Referans embeddingleri hesapla
print("🔄 Referans resimlerin embeddingleri hesaplanıyor...")
reference_embeddings, reference_paths = encode_images_batch(first50_files)
print(f"✅ {len(reference_paths)} referans resim encode edildi")
print(f"📐 Embeddings shape: {reference_embeddings.shape}")

# Referans bilgilerini kaydet
reference_names = [p.name for p in reference_paths]
reference_df = pd.DataFrame({
    "filename": reference_names,
    "path": [str(p) for p in reference_paths]
})

# Embeddings'i GPU'da tut (hızlı karşılaştırma için)
reference_embeddings_gpu = reference_embeddings.to(device)

## 7. Raw Images Klasör Analizi

In [ ]:
# raw-images altındaki tüm klasörleri tara
print("🔍 raw-images klasörleri taranıyor...")

folder_counts = {}
folders = sorted([f for f in RAW_IMAGES_PATH.iterdir() if f.is_dir()])

for folder in tqdm(folders, desc="Klasörler taranıyor"):
    image_files = get_image_files(folder)
    folder_counts[folder.name] = len(image_files)

# DataFrame olarak göster
folder_df = pd.DataFrame([
    {"folder": k, "image_count": v} for k, v in folder_counts.items()
]).sort_values("image_count", ascending=False)

print(f"\n📊 Toplam {len(folder_counts)} klasör bulundu")
print(f"📷 Toplam resim sayısı: {folder_df['image_count'].sum():,}")
print("\n" + "="*50)
folder_df

In [ ]:
# folder_counts dictionary'sini kaydet
folder_counts_path = BASE_PATH / "folder_counts.csv"
folder_df.to_csv(folder_counts_path, index=False)
print(f"💾 Klasör sayıları kaydedildi: {folder_counts_path}")

## 8. Örnekleme Hedeflerini Belirle

Aşağıdaki hücrede her klasörden kaç resim işleneceğini belirleyin.  
`TARGET_COUNTS` dictionary'sine klasör adı ve sayı ekleyin.

In [ ]:
# ========== ÖRNEKLEME HEDEFLERİ ==========
# Her klasörden kaç resim işleneceğini belirle
# Belirtilmeyen klasörler için DEFAULT_SAMPLE_COUNT kullanılır

TARGET_COUNTS = {
    # Örnekler (klasör isimlerinize göre düzenleyin):
    # "bedroom": 2000,
    # "kitchen": 1500,
    # "bathroom": 1000,
}

# İşlenecek klasörleri ve sayıları hesapla
sampling_plan = []
for folder_name, total_count in folder_counts.items():
    target = TARGET_COUNTS.get(folder_name, DEFAULT_SAMPLE_COUNT)
    actual = min(target, total_count)  # Klasördeki resimden fazla alamayız
    sampling_plan.append({
        "folder": folder_name,
        "total": total_count,
        "target": target,
        "actual": actual
    })

sampling_df = pd.DataFrame(sampling_plan).sort_values("actual", ascending=False)
total_to_process = sampling_df["actual"].sum()

# A100 ile tahmini süre (batch=256, ~500-800 img/s)
estimated_speed = 600  # A100 için ortalama img/s
estimated_minutes = total_to_process / estimated_speed / 60

print(f"📋 Örnekleme Planı:")
print(f"   Toplam işlenecek resim: {total_to_process:,}")
print(f"   Tahmini süre (A100 ~{estimated_speed} img/s): ~{estimated_minutes:.1f} dakika")
print("\n" + "="*50)
sampling_df

## 9. Ana Filtreleme Döngüsü 🚀

In [ ]:
# Ana filtreleme fonksiyonu
def process_folder(folder_name: str, sample_count: int) -> list:
    """
    Tek bir klasörü işle:
    1. Resimleri örnekle
    2. Embeddinglari hesapla
    3. Referanslarla karşılaştır
    4. Threshold ALTINDA olanları kaydet (benzemeyen resimler)

    Returns: List of result dicts
    """
    folder_path = RAW_IMAGES_PATH / folder_name
    output_folder = FILTERED_PATH / folder_name
    output_folder.mkdir(parents=True, exist_ok=True)

    # Resimleri al ve örnekle
    all_images = get_image_files(folder_path)
    if len(all_images) > sample_count:
        sampled_images = random.sample(all_images, sample_count)
    else:
        sampled_images = all_images

    if not sampled_images:
        return []

    # Embeddinglari hesapla
    embeddings, valid_paths = encode_images_batch(sampled_images)

    if len(valid_paths) == 0:
        return []

    # Similarity hesapla
    embeddings_gpu = embeddings.to(device)
    max_scores, best_indices = compute_similarity(embeddings_gpu, reference_embeddings_gpu)

    # Threshold ALTINDA olanları filtrele (benzemeyen resimler)
    results = []
    filtered_count = 0

    for i, (path, score, ref_idx) in enumerate(zip(valid_paths, max_scores, best_indices)):
        score_val = score.item()
        ref_idx_val = ref_idx.item()

        # TERS MANTIK: Benzerlik threshold'un ALTINDAYSA al (benzemeyen resim)
        if score_val < SIMILARITY_THRESHOLD:
            # Resmi kopyala
            dest_path = output_folder / path.name
            shutil.copy2(path, dest_path)
            filtered_count += 1

            results.append({
                "source_path": str(path),
                "dest_path": str(dest_path),
                "folder": folder_name,
                "filename": path.name,
                "similarity_score": score_val,
                "best_ref_idx": ref_idx_val,
                "best_ref_name": reference_names[ref_idx_val]
            })

    return results


print("✅ Filtreleme fonksiyonu hazır (benzemeyen resimler alınacak)")

In [ ]:
# ========== TÜM KLASÖRLERİ İŞLE ==========
print("🚀 Filtreleme başlıyor (benzemeyen resimler alınacak)...")
print(f"   Threshold: < {SIMILARITY_THRESHOLD} (bu değerden DÜŞÜK olanlar alınacak)")
print(f"   Toplam klasör: {len(sampling_df)}")
print("="*60)

all_results = []
folder_stats = []

for _, row in tqdm(sampling_df.iterrows(), total=len(sampling_df), desc="Klasörler"):
    folder_name = row["folder"]
    sample_count = row["actual"]

    print(f"\n📂 {folder_name} ({sample_count} resim)...")

    results = process_folder(folder_name, sample_count)
    all_results.extend(results)

    folder_stats.append({
        "folder": folder_name,
        "processed": sample_count,
        "filtered": len(results),
        "rate": len(results) / sample_count * 100 if sample_count > 0 else 0
    })

    print(f"   ✅ {len(results)} benzemeyen resim bulundu (similarity < {SIMILARITY_THRESHOLD})")

print("\n" + "="*60)
print(f"🎉 Filtreleme tamamlandı!")
print(f"   Toplam işlenen: {sum(s['processed'] for s in folder_stats):,}")
print(f"   Toplam benzemeyen resim: {len(all_results):,}")

## 10. Sonuçları Kaydet

In [ ]:
# Sonuçları DataFrame'e çevir ve kaydet
results_df = pd.DataFrame(all_results)
stats_df = pd.DataFrame(folder_stats)

# Parquet olarak kaydet (daha verimli)
results_path = BASE_PATH / "filtered_manifest.parquet"
results_df.to_parquet(results_path, index=False)
print(f"💾 Manifest kaydedildi: {results_path}")

# CSV olarak da kaydet (okunabilirlik için)
results_csv_path = BASE_PATH / "filtered_manifest.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"💾 CSV kaydedildi: {results_csv_path}")

# İstatistikleri kaydet
stats_path = BASE_PATH / "filtering_stats.csv"
stats_df.to_csv(stats_path, index=False)
print(f"💾 İstatistikler kaydedildi: {stats_path}")

In [ ]:
# Referans embeddingleri kaydet
ref_embeddings_path = BASE_PATH / "reference_embeddings.npy"
np.save(ref_embeddings_path, reference_embeddings.numpy())
print(f"💾 Referans embeddingler kaydedildi: {ref_embeddings_path}")

# Referans manifest
ref_manifest_path = BASE_PATH / "reference_manifest.csv"
reference_df.to_csv(ref_manifest_path, index=False)
print(f"💾 Referans manifest kaydedildi: {ref_manifest_path}")

## 11. Özet & İstatistikler

In [ ]:
# Genel özet
print("="*60)
print("📊 FİLTRELEME ÖZETİ")
print("="*60)
print(f"Referans resim sayısı: {len(reference_paths)}")
print(f"İşlenen klasör sayısı: {len(folder_stats)}")
print(f"Toplam işlenen resim: {stats_df['processed'].sum():,}")
print(f"Toplam filtrelenen resim: {len(all_results):,}")
print(f"Genel geçiş oranı: {len(all_results) / stats_df['processed'].sum() * 100:.2f}%")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")
print("="*60)

# Klasör bazlı istatistikler
print("\n📈 Klasör Bazlı İstatistikler:")
stats_df_sorted = stats_df.sort_values("filtered", ascending=False)
stats_df_sorted

In [ ]:
# Similarity score dağılımı
if len(results_df) > 0:
    print("\n📉 Similarity Score Dağılımı:")
    print(results_df["similarity_score"].describe())

    # En çok eşleşen referans resimler
    print("\n🏆 En Çok Eşleşen Referans Resimler:")
    ref_counts = results_df["best_ref_name"].value_counts().head(10)
    for name, count in ref_counts.items():
        print(f"   {name}: {count} eşleşme")

## 12. Görsel Doğrulama (Sanity Check)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

def show_sample_matches(n_samples=5):
    """Rastgele filtrelenmiş resimleri ve eşleştikleri referansları göster"""
    if len(results_df) == 0:
        print("❌ Filtrelenmiş resim yok!")
        return

    # Rastgele örnekler seç
    samples = results_df.sample(min(n_samples, len(results_df)))

    fig, axes = plt.subplots(n_samples, 2, figsize=(10, 4*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)

    for idx, (_, row) in enumerate(samples.iterrows()):
        # Filtrelenmiş resim
        try:
            filtered_img = Image.open(row["dest_path"])
            axes[idx, 0].imshow(filtered_img)
            axes[idx, 0].set_title(f"Filtered: {row['filename']}\nScore: {row['similarity_score']:.3f}")
            axes[idx, 0].axis("off")
        except:
            axes[idx, 0].set_title("Resim yüklenemedi")
            axes[idx, 0].axis("off")

        # Eşleşen referans resim
        try:
            ref_path = reference_paths[row["best_ref_idx"]]
            ref_img = Image.open(ref_path)
            axes[idx, 1].imshow(ref_img)
            axes[idx, 1].set_title(f"Reference: {row['best_ref_name']}")
            axes[idx, 1].axis("off")
        except:
            axes[idx, 1].set_title("Referans yüklenemedi")
            axes[idx, 1].axis("off")

    plt.tight_layout()
    plt.show()

# 5 örnek göster
show_sample_matches(5)

## 13. Opsiyonel: Farklı Threshold ile Yeniden Çalıştır

Eğer sonuçlar çok fazla veya çok az resim içeriyorsa, threshold'u değiştirip sadece karşılaştırma kısmını yeniden çalıştırabilirsiniz.

In [ ]:
# Eğer manifest zaten varsa, farklı threshold ile filtreleme yapabilirsiniz
# Bu hücre opsiyoneldir - sadece threshold değiştirmek istediğinizde çalıştırın

NEW_THRESHOLD = 0.65  # Yeni threshold (daha düşük = daha farklı resimler)

# Mevcut sonuçlardan yeni threshold ile filtrele
if len(results_df) > 0:
    # Yeni threshold'dan DÜŞÜK olanları al
    new_filtered = results_df[results_df["similarity_score"] < NEW_THRESHOLD]
    print(f"Threshold < {SIMILARITY_THRESHOLD} -> < {NEW_THRESHOLD}")
    print(f"Resim sayısı: {len(results_df)} -> {len(new_filtered)}")
    print(f"Değişim: {len(new_filtered) - len(results_df)} resim")

---

## 📋 Kullanım Kılavuzu

1. **Kurulum**: İlk 3 hücreyi çalıştırın (pip install, imports, drive mount)
2. **Konfigürasyon**:
   - `TARGET_COUNTS` dictionary'sine klasör isimlerini ve alınacak resim sayılarını girin
   - `SIMILARITY_THRESHOLD` değerini ayarlayın
3. **Referans resimler**: `first-50/` klasörüne referans resimlerinizi koyun
4. **Çalıştır**: Tüm hücreleri sırayla çalıştırın
5. **Sonuçlar**:
   - `filtered-images/` klasöründe benzemeyen resimler
   - `filtered_manifest.csv/parquet` dosyasında detaylar
   - `filtering_stats.csv` dosyasında klasör bazlı istatistikler

### ⚠️ Threshold Mantığı (TERS):
Bu notebook **benzemeyen** resimleri buluyor. Yani:
- Similarity score **threshold'un ALTINDA** ise resim alınır
- `SIMILARITY_THRESHOLD = 0.70` → similarity < 0.70 olan resimler alınır

### Threshold Rehberi:
- **< 0.50**: Çok farklı resimler (referansla alakasız)
- **< 0.60**: Farklı resimler
- **< 0.70**: Orta düzey farklılık (önerilen başlangıç)
- **< 0.80**: Hafif farklı resimler

### 🚀 A100 Performans Notları:
- **Batch Size**: 256 (A100 40GB için optimal, 80GB için 512 denenebilir)
- **FP16 Mode**: Model half precision'da çalışıyor (2x hız)
- **TF32**: Tensor Float 32 aktif (matmul hızlandırma)
- **Beklenen Hız**: ~500-800 img/s (Drive IO'ya bağlı)
- **Tip**: Drive IO darboğazı için resimleri `/content/` altına kopyalayabilirsiniz